# Two-Layer Teacher Temperature LR Sweep

Loads observable-only sweep CSVs and plots Pareto curves plus time-series diagnostics.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from two_layer_experiment.results import best_learning_rates, load_results
from two_layer_experiment.plotting import pareto_plot

runs_dir = Path('../runs/two_layer_lr_sweep')
df = load_results(runs_dir)
best = best_learning_rates(df)
best

In [ ]:
for mode in ['minibatch32', 'population']:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    pareto_plot(df, mode, ax=ax)
    fig.tight_layout()
    fig.savefig(runs_dir / f'pareto_{mode}.png', dpi=180)
    plt.show()

In [ ]:
def best_lr_timeseries(df, best, mode):
    rows = []
    for _, row in best[best['mode'] == mode].iterrows():
        rows.append(df[(df['mode'] == mode) & (df['beta'] == row['beta']) & (df['lr'] == row['lr'])])
    return __import__('pandas').concat(rows, ignore_index=True)

def plot_metric_over_time(ts, metric, mode):
    fig, ax = plt.subplots(figsize=(9, 5.2))
    for beta, group in ts.groupby('beta', sort=True):
        group = group.sort_values('step')
        ax.plot(group['step'], group[metric], marker='o', linewidth=1.5, label=f'beta={beta:g}')
    ax.set_title(f'{mode}: {metric} at best final-xent LR')
    ax.set_xlabel('SGD step')
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.25)
    ax.legend(ncols=2, fontsize='small')
    fig.tight_layout()
    fig.savefig(runs_dir / f'time_{mode}_{metric}.png', dpi=180)
    return fig

time_metrics = [
    'teacher_h1_rms',
    'teacher_h2_rms',
    'student_h1_rms',
    'student_h2_rms',
    'student_df_dh1_rms',
    'population_xent',
    'population_logit_mse',
    'w1_cosine_fro',
]

In [ ]:
for mode in ['minibatch32', 'population']:
    ts = best_lr_timeseries(df, best, mode)
    for metric in time_metrics:
        fig = plot_metric_over_time(ts, metric, mode)
        plt.show()